# Authorship analysis — temporal dynamics

## Overview

This notebook extends the corpus-wide authorship overlap analysis from `07_authorship.ipynb` with two temporal dimensions:

- **Part B: Year-by-year same-conference overlap.** Restricts the overlap question to authors who published at *both* AGILE and GIScience *in the same calendar year*. GIScience off-years (2017, 2019, 2022, 2024) correctly yield zero by construction.
- **Section 10: Reproducibility of cross-conference-author papers.** Examines whether papers written by authors who publish at both venues track, lead, or lag the overall corpus trend in reproducibility levels — both as a bubble-matrix over time and as a pre-vs-post-intervention delta table.
- **Section 11: Cross-group author overlap (non-independence check).** Raw view of how many papers in the four analytical groups (`agile_pre`, `agile_post`, `giscience_pre`, `giscience_post`) share at least one author, to contextualise the independence assumption made in `04_results_hypotheses.ipynb`.

**Inputs** (read-only; produced by `07_authorship.ipynb`):
- `data-clean/authors.csv` — authoritative per-author-per-paper table with resolved identities.
- `data-clean/all-data.csv` — paper-level reproducibility assessments.

**Outputs** written to `data/`:
- `authors-cross-conference-yearly.csv` — per-year same-year overlap counts.
- `cross-author-reprolevels-yearly.csv` — per-(year, level, subset) reproducibility counts.
- `cross-author-intervention-delta.csv` — pre-vs-post %HIGH delta for all vs. cross-author papers.
- `cross-group-overlap-multipaper-authors.csv` — raw author-paper table for the non-independence check.

**Note**: This notebook was largely co-edited with Claude Code. All code snippets and data were inspected thoroughly by a contributing author and checked for correctness.

## 1. Setup

In [ ]:
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

FIG_DIR = Path("figs")
FIG_DIR.mkdir(exist_ok=True)

pd.set_option("display.max_colwidth", 120)

## 2. Load inputs

Both inputs are produced by `07_authorship.ipynb` and must be regenerated there before re-running this notebook.

In [ ]:
authors_df = pd.read_csv("data-clean/authors.csv")
print(f"authors_df: {len(authors_df)} rows, {authors_df['paper'].nunique()} papers, "
      f"{authors_df['identity'].nunique()} distinct identities")

df_all = pd.read_csv("data-clean/all-data.csv")
df = df_all[df_all["consolidated_cp"] == False].copy().reset_index(drop=True)
print(f"non-conceptual papers: {len(df)} "
      f"(AGILE: {(df['conf']=='agile').sum()}, GIScience: {(df['conf']=='giscience').sum()})")

## 3. Part B — Year-by-year same-year overlap

Part A in `07_authorship.ipynb` pooled all years together, so an AGILE author from 2019 and a GIScience author from 2023 could be counted as overlapping. That is the right framing for *community overlap*, but misleading for *temporal dynamics*: it "inherits" overlap from every other year, and it silently invents overlap in GIScience off-years (2017, 2019, 2022, 2024), which had no GIScience conference at all.

Part B constrains the question to a single year at a time: **in year *y*, how many authors published at both AGILE and GIScience *that year*?** GIScience off-years must be zero by construction.

### B.1 Shared-year authors and affected papers

In [ ]:
# Universe of years present in the corpus.
corpus_years = sorted(authors_df["year"].unique())

# Years each conference actually met.
agile_years = set(authors_df[authors_df["conf"] == "agile"]["year"].unique())
gis_years = set(authors_df[authors_df["conf"] == "giscience"]["year"].unique())
print("AGILE years:    ", sorted(agile_years))
print("GIScience years:", sorted(gis_years))
shared_years = sorted(agile_years & gis_years)
print("Shared years (both conferences met):", shared_years)

# Per-year identity sets, per conference.
yearly_rows = []
for year in corpus_years:
    ag_set = set(
        authors_df[(authors_df["conf"] == "agile") & (authors_df["year"] == year)]["identity"]
    )
    gi_set = set(
        authors_df[(authors_df["conf"] == "giscience") & (authors_df["year"] == year)]["identity"]
    )
    shared = ag_set & gi_set

    ag_papers = authors_df[(authors_df["conf"] == "agile") & (authors_df["year"] == year)]
    gi_papers = authors_df[(authors_df["conf"] == "giscience") & (authors_df["year"] == year)]

    ag_cross_papers = ag_papers[ag_papers["identity"].isin(shared)]["paper"].nunique()
    gi_cross_papers = gi_papers[gi_papers["identity"].isin(shared)]["paper"].nunique()
    ag_total = ag_papers["paper"].nunique()
    gi_total = gi_papers["paper"].nunique()

    yearly_rows.append(
        {
            "year": year,
            "shared_authors": len(shared),
            "agile_papers": ag_total,
            "giscience_papers": gi_total,
            "agile_cross_papers": ag_cross_papers,
            "giscience_cross_papers": gi_cross_papers,
            "agile_cross_share": (ag_cross_papers / ag_total) if ag_total else 0.0,
            "giscience_cross_share": (gi_cross_papers / gi_total) if gi_total else 0.0,
            "both_met": (year in agile_years) and (year in gis_years),
        }
    )

yearly_overlap = pd.DataFrame(yearly_rows)
display(yearly_overlap)

### B.2 Figure: Papers with a same-year cross-conference author

Two directions, plotted separately:

- **AGILE to GIScience:** AGILE papers in year *y* for which at least one author *also*
  co-authored a GIScience paper in year *y*.
- **GIScience to AGILE:** GIScience papers in year *y* for which at least one author *also*
  co-authored an AGILE paper in year *y*.

In years without a GIScience conference (2017, 2019, 2022, 2024) both series are zero by construction.

Styling follows the convention used in the R/Quarto notebooks (Brewer `Set1`, Tufte-style minimal theme).

In [ ]:
from matplotlib.ticker import MaxNLocator

SET1 = plt.get_cmap("Set1").colors
AG_LABEL = "AGILE papers with a same-year GIScience-also author"
GI_LABEL = "GIScience papers with a same-year AGILE-also author"
dir_colors = {AG_LABEL: SET1[0], GI_LABEL: SET1[1]}

fig, (ax_n, ax_p) = plt.subplots(1, 2, figsize=(12, 4.8), sharex=True)

# Drop years where the structural overlap is undefined (one of the conferences
# did not run that year). The grey bands still mark those years explicitly.
yo = yearly_overlap[yearly_overlap["both_met"]].sort_values("year")
off_years = [
    int(y) for y in sorted(yearly_overlap["year"])
    if not yearly_overlap.loc[yearly_overlap["year"] == y, "both_met"].iloc[0]
]

# Markers only -- no connecting lines, since adjacent valid years can be far apart.
ax_n.plot(yo["year"], yo["agile_cross_papers"],
          marker="o", linestyle="", color=dir_colors[AG_LABEL], label=AG_LABEL)
ax_n.plot(yo["year"], yo["giscience_cross_papers"],
          marker="o", linestyle="", color=dir_colors[GI_LABEL], label=GI_LABEL)

ax_p.plot(yo["year"], yo["agile_cross_share"] * 100,
          marker="o", linestyle="", color=dir_colors[AG_LABEL], label=AG_LABEL)
ax_p.plot(yo["year"], yo["giscience_cross_share"] * 100,
          marker="o", linestyle="", color=dir_colors[GI_LABEL], label=GI_LABEL)

all_years = sorted(yearly_overlap["year"].unique())
for ax in (ax_n, ax_p):
    for y in off_years:
        ax.axvspan(y - 0.4, y + 0.4, color="lightgrey", alpha=0.25, zorder=0)
    ax.set_xlabel("Conference year")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(True, axis="y", linestyle=":", alpha=0.6)
    ax.set_xticks(all_years)
    ax.legend(frameon=False, loc="upper left", fontsize=9)

ax_n.set_ylim(0, 4)
ax_n.yaxis.set_major_locator(MaxNLocator(integer=True))
ax_n.set_ylabel("Papers with >=1 same-year author at the other venue")
ax_n.set_title("Absolute count")

ax_p.set_ylim(0, 50)
ax_p.set_ylabel("Share of the year's papers (%)")
ax_p.set_title("Share per year")

# Subtitle with corpus extent (computational == non-conceptual papers).
year_min = int(df["year"].min())
year_max = int(df["year"].max())
subtitle = (
    f"Conference years {year_min}--{year_max} | "
    f"{len(df)} non-conceptual papers "
    f"(AGILE n={(df['conf']=='agile').sum()}, GIScience n={(df['conf']=='giscience').sum()})"
)
fig.suptitle(
    "Cross-conference authorship -- restricted to same-year overlap\n"
    + subtitle
    + "\n(shaded bands mark years without a GIScience conference)",
    fontsize=11,
)
fig.tight_layout()
fig.savefig(FIG_DIR / "figure_authorship_cross_conf_timeline.png", dpi=150)
fig.savefig(FIG_DIR / "figure_authorship_cross_conf_timeline.pdf")
plt.show()

## 4. Reproducibility for cross-conference-author papers

Two complementary views of whether papers authored by people who publish at *both* AGILE and GIScience track the overall corpus trend, lead it, or lag behind. Throughout this section a paper counts as *cross-author* if at least one of its authors (resolved identity, pooling all years in the corpus) also appears on at least one paper at the other conference series -- the time-agnostic, direction-pooled definition from Part A in `07_authorship.ipynb`. Direction (AGILE-to-GIScience vs. GIScience-to-AGILE) is deliberately ignored.

### 4.1 Per-year reproducibility-level distribution

A bubble-matrix plot per criterion (*Input Data*, *Methods*, *Results*): each year shows one column of four circles (one per UDAO level on the y-axis), with circle area proportional to the number of papers at that level in that year. Both subsets below **pool AGILE and GIScience papers** (a year's column therefore combines all papers published at either conference that year). Two subsets are overlaid on the same axes:

- **Grey hollow outlines**: all non-conceptual papers in the corpus (both AGILE and
  GIScience pooled) -- the "overall" backdrop.
- **Red filled circles**: cross-author papers only, again pooled across the two
  conferences. Because cross-author papers are a strict subset of overall, the red
  circle is always at or inside the grey one; the visible gap is the "non-cross-author"
  share for that (year, level) cell.

Higher rows (A, O) mark more reproducible papers. Reading the plot vertically in each year shows how that year's papers are distributed across levels; reading horizontally along a level shows how that level's headcount changes over time. The per-(year, level, subset) counts are also persisted to
[`data/cross-author-reprolevels-yearly.csv`](data/cross-author-reprolevels-yearly.csv) for downstream use.

In [ ]:
from matplotlib.lines import Line2D

LEVEL_LABELS = ["U", "D", "A", "O"]  # y-axis: bottom (U) -> top (O), increasing reproducibility
CRITERIA = ["data", "methods", "results"]
CRIT_TITLE = {"data": "Input Data", "methods": "Methods", "results": "Results"}

# Cross-conference identities (any year; direction pooled, matching Part A).
cross_ids = (
    set(authors_df[authors_df["conf"] == "agile"]["identity"]) &
    set(authors_df[authors_df["conf"] == "giscience"]["identity"])
)
papers_with_cross = set(authors_df[authors_df["identity"].isin(cross_ids)]["paper"])

repro = df[["paper", "conf", "year"] + [f"consolidated_{c}" for c in CRITERIA]].copy()
repro["is_cross"] = repro["paper"].isin(papers_with_cross)


def counts_by_year_level(sub: pd.DataFrame, crit: str) -> pd.DataFrame:
    if len(sub) == 0:
        return pd.DataFrame(columns=["year", "level", "n"])
    return (
        sub.groupby(["year", f"consolidated_{crit}"]).size()
        .reset_index(name="n")
        .rename(columns={f"consolidated_{crit}": "level"})
    )


SET1 = plt.get_cmap("Set1").colors
COLOR_CROSS = SET1[0]
COLOR_OVERALL = "0.35"
SIZE_BASE = 25          # base marker area (pt^2) so a 1-paper dot is still visible
SIZE_PER_PAPER = 30     # additional area per paper (linear in count)

fig, axes = plt.subplots(1, 3, figsize=(14, 4.8), sharey=True)
years = sorted(repro["year"].unique())
for ax, crit in zip(axes, CRITERIA):
    overall = counts_by_year_level(repro, crit)
    cross   = counts_by_year_level(repro[repro["is_cross"]], crit)

    # Grey hollow outline = all papers. Drawn first, underneath.
    for _, r in overall.iterrows():
        y = LEVEL_LABELS.index(r["level"])
        ax.scatter(r["year"], y, s=SIZE_BASE + SIZE_PER_PAPER * r["n"],
                   facecolor="none", edgecolor=COLOR_OVERALL, linewidth=1.1, zorder=3)
    # Red filled = cross-author subset. Always <= the grey outline at the same cell.
    for _, r in cross.iterrows():
        y = LEVEL_LABELS.index(r["level"])
        ax.scatter(r["year"], y, s=SIZE_BASE + SIZE_PER_PAPER * r["n"],
                   facecolor=COLOR_CROSS, edgecolor="white", linewidth=0.5,
                   alpha=0.85, zorder=5)

    ax.set_title(CRIT_TITLE[crit])
    ax.set_xlabel("Conference year")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(True, linestyle=":", alpha=0.4)
    ax.set_xticks(years)
    ax.set_ylim(-0.7, 3.7)

axes[0].set_yticks(range(4))
axes[0].set_yticklabels(LEVEL_LABELS)
axes[0].set_ylabel("Reproducibility level")

subset_hints = [
    Line2D([0], [0], marker="o", linestyle="", markersize=9,
           markerfacecolor="none", markeredgecolor=COLOR_OVERALL,
           label="All papers"),
    Line2D([0], [0], marker="o", linestyle="", markersize=9,
           markerfacecolor=COLOR_CROSS, markeredgecolor="white",
           label="Cross-conference-author papers"),
]
axes[0].legend(handles=subset_hints, loc="upper left", fontsize=9, frameon=False)

size_hints = [
    Line2D([0], [0], marker="o", linestyle="", markersize=(SIZE_BASE + SIZE_PER_PAPER * n) ** 0.5,
           markerfacecolor="lightgrey", markeredgecolor=COLOR_OVERALL, label=f"  {n}")
    for n in (1, 5, 15, 30)
]
axes[-1].legend(handles=size_hints, loc="lower right", fontsize=8, frameon=False,
                title="papers", labelspacing=1.4, borderpad=1.0)

subtitle = (
    f"Conference years {int(repro['year'].min())}--{int(repro['year'].max())} | "
    f"{len(repro)} non-conceptual papers (AGILE + GIScience pooled); "
    f"{int(repro['is_cross'].sum())} with >=1 cross-conference author "
    f"({len(cross_ids)} such authors in total; direction pooled)"
)
fig.suptitle(
    "Reproducibility level distribution over time -- cross-conference-author papers vs. all papers\n"
    + subtitle, fontsize=11,
)
fig.tight_layout()
fig.savefig(FIG_DIR / "figure_authorship_cross_author_reprolevels.png", dpi=150)
fig.savefig(FIG_DIR / "figure_authorship_cross_author_reprolevels.pdf")
plt.show()

# Persist per-(year, level, subset) counts for downstream use.
rows = []
for crit in CRITERIA:
    for subset_name, sub in (("all-papers", repro), ("cross-author", repro[repro["is_cross"]])):
        c = counts_by_year_level(sub, crit)
        for _, r in c.iterrows():
            rows.append({"criterion": CRIT_TITLE[crit], "subset": subset_name,
                         "year": int(r["year"]), "level": r["level"], "n": int(r["n"])})
pd.DataFrame(rows).to_csv("data/cross-author-reprolevels-yearly.csv", index=False)
print("wrote data/cross-author-reprolevels-yearly.csv")

### 4.2 Absolute change in *HIGH* levels across the intervention

Mirrors the gt-table *"GIScience vs AGILE: Absolute change in percentage points"* from [`03_results_reprolevels.qmd`](03_results_reprolevels.qmd) (manuscript Table showing the pre-vs-post intervention shift), but adds a second "cross-author only" view beside each conference so we can ask: **are cross-conference authors leading the improvement, or lagging behind?**

**Definition.** Each criterion is aggregated to a binary level: `HIGH = {A, O}` (the two higher reproducibility levels) vs. `LOW = {U, D}`. For each conference the corpus is split at the intervention year: *pre* = `year < 2020`, *post* = `year > 2020`; 2020 is excluded (transition year -- the AGILE guidelines were optional). The statistic is the per-conference per-criterion delta `%HIGH(post) - %HIGH(pre)`, in percentage points.

**Reading the table.** For each conference we report the delta for **all papers** (as in the original table) and for the **cross-author subset** side by side. The last column, *lead/lag*, is `delta(cross) - delta(all)`: positive means cross-authors are pulling the curve up faster than the overall corpus on that criterion; negative means they trail. Sample sizes (`n_pre`, `n_post`) are reported because the cross-author subset is small for GIScience pre (13 papers).

In [ ]:
INTERVENTION_YEAR = 2020  # 2020 is excluded (AGILE transition year)
HIGH_LEVELS = {"A", "O"}


def pct_high(sub: pd.DataFrame, col: str) -> tuple[float, int]:
    """Return (%HIGH, N) for a subset of papers on one consolidated_{crit} column."""
    if len(sub) == 0:
        return float("nan"), 0
    n_high = int(sub[col].isin(HIGH_LEVELS).sum())
    return n_high / len(sub), len(sub)


rows = []
for conf in ("giscience", "agile"):
    base = df[df["conf"] == conf]
    for subset_name, subset in (
        ("all papers", base),
        ("cross-author only", base[base["paper"].isin(papers_with_cross)]),
    ):
        pre  = subset[subset["year"] < INTERVENTION_YEAR]
        post = subset[subset["year"] > INTERVENTION_YEAR]
        for crit in CRITERIA:
            col = f"consolidated_{crit}"
            p_pre,  n_pre  = pct_high(pre,  col)
            p_post, n_post = pct_high(post, col)
            rows.append({
                "criterion": CRIT_TITLE[crit],
                "conf": conf.upper(),
                "subset": subset_name,
                "pct_HIGH_pre":  round(p_pre  * 100, 1) if n_pre  else float("nan"),
                "n_pre":  n_pre,
                "pct_HIGH_post": round(p_post * 100, 1) if n_post else float("nan"),
                "n_post": n_post,
                "delta_pp": round((p_post - p_pre) * 100, 1),
            })
delta_tbl = pd.DataFrame(rows)

# Side-by-side pivot matching the manuscript's table layout, with cross-author column added.
pivot = (
    delta_tbl.set_index(["criterion", "conf", "subset"])["delta_pp"]
    .unstack(["conf", "subset"])
    .reindex(["Input Data", "Methods", "Results"])
    [[("GISCIENCE", "all papers"), ("GISCIENCE", "cross-author only"),
      ("AGILE",     "all papers"), ("AGILE",     "cross-author only")]]
)
pivot.columns = pd.MultiIndex.from_tuples(
    [("GIScience", "all"), ("GIScience", "cross"),
     ("AGILE",     "all"), ("AGILE",     "cross")],
    names=["conference", "subset"],
)
print("Absolute change in %HIGH across the intervention (pp):")
print(pivot.to_string())

# Lead/lag column: cross - all (per conference x criterion).
lead_lag = (
    delta_tbl.set_index(["criterion", "conf", "subset"])["delta_pp"]
    .unstack("subset")
    .assign(**{"lead/lag (pp)": lambda d: d["cross-author only"] - d["all papers"]})
    .reindex(["Input Data", "Methods", "Results"], level=0)
    [["all papers", "cross-author only", "lead/lag (pp)"]]
    .round(1)
)
print()
print("Lead/lag of the cross-author subset (positive = leading, negative = lagging):")
print(lead_lag.to_string())

print()
print("Sample sizes (cross-author subset only):")
sizes = (
    delta_tbl[delta_tbl["subset"] == "cross-author only"]
    [["criterion", "conf", "n_pre", "n_post"]]
    .drop_duplicates()
    .reset_index(drop=True)
)
print(sizes.to_string(index=False))

delta_tbl.to_csv("data/cross-author-intervention-delta.csv", index=False)
print()
print("wrote data/cross-author-intervention-delta.csv")

### 4.3 Figure: reproducibility by criterion, faceted by group

Small multiples -- one panel per analytical group (`agile_pre`, `agile_post`, `giscience_pre`, `giscience_post`). Each point is **one distinct paper** in the
group, positioned at (reproducibility criterion, achieved level) with uniform random jitter on both axes so overlapping points become visible. Two subsets are overlaid:

- **Toned-down grey** -- papers whose authors do *not* publish in any of the other three groups. Drawn underneath as a low-contrast backdrop so the eye sees the shape of the full analytical group. All grey dots have the same size.
- **Coloured (one colour per group)** -- papers with at least one "cross-group author", i.e. an author who also publishes in one of the other three groups.
  Drawn on top so the subset that drives non-independence stands out. **Circle area scales with the number of cross-group authors on that paper**, so papers with many such authors are visually heavier -- they are the strongest source of non-independence.

Each panel title reports the number of cross-group-author papers together with the total number of papers in the analytical group and the resulting percentage. A direct read-out of how much non-independent signal each group carries into the main pre-vs-post and AGILE-vs-GIScience comparisons.

In [ ]:
# Build the four analytical groups (agile_pre, agile_post, giscience_pre, giscience_post).
# 2020 is excluded -- AGILE transition year, excluded project-wide.
INTERVENTION_YEAR = 2020


def _group_of(year: int, conf: str) -> str | None:
    if year < INTERVENTION_YEAR:
        return f"{conf}_pre"
    if year > INTERVENTION_YEAR:
        return f"{conf}_post"
    return None  # 2020 excluded


grouped = authors_df.copy()
grouped["group"] = [
    _group_of(int(y), c) for y, c in zip(grouped["year"], grouped["conf"])
]
grouped = grouped.dropna(subset=["group"])

In [ ]:
from matplotlib.lines import Line2D
from matplotlib.patches import Rectangle

GROUPS = ["agile_pre", "agile_post", "giscience_pre", "giscience_post"]
GROUP_TITLES = {
    "agile_pre":      "AGILE pre (year < 2020)",
    "agile_post":     "AGILE post (year > 2020)",
    "giscience_pre":  "GIScience pre (year < 2020)",
    "giscience_post": "GIScience post (year > 2020)",
}
CRITERIA = ["data", "methods", "results"]
CRIT_LABELS = {"data": "Input Data", "methods": "Methods", "results": "Results"}
LEVEL_LABELS = ["U", "D", "A", "O"]  # bottom -> top, increasing reproducibility

SET1 = plt.get_cmap("Set1").colors
GROUP_COLORS = {
    "agile_pre":      SET1[1],  # blue   (AGILE pre)
    "agile_post":     SET1[0],  # red    (AGILE post)
    "giscience_pre":  SET1[2],  # green  (GIScience pre)
    "giscience_post": SET1[4],  # orange (GIScience post)
}
COLOR_NONCROSS = "0.55"
BAND_COLOR = "0.92"
SIZE_BASE = 24
SIZE_PER_AUTHOR = 28

rng = np.random.default_rng(20260416)
X_JIT = 0.25
Y_JIT = 0.18
BAND_HALF = 0.38

# Cross-group authors = identities appearing in >= 2 of the 4 groups.
author_groups = grouped.groupby("identity")["group"].agg(lambda s: set(s))
crossgroup_ids = set(author_groups[author_groups.map(len) >= 2].index)

crossgroup_author_counts = (
    grouped[grouped["identity"].isin(crossgroup_ids)]
    .groupby(["group", "paper"])["identity"]
    .nunique()
    .rename("n_crossgroup_authors")
    .reset_index()
)

all_group_papers = (
    grouped[["group", "paper", "year"]].drop_duplicates()
    .merge(
        df[["paper", "consolidated_data", "consolidated_methods",
            "consolidated_results"]],
        on="paper", how="left",
    )
    .merge(crossgroup_author_counts, on=["group", "paper"], how="left")
)
assert (all_group_papers["year"] != 2020).all(), \
    "Figure must not include papers from the transitional year 2020"
all_group_papers["n_crossgroup_authors"] = (
    all_group_papers["n_crossgroup_authors"].fillna(0).astype(int)
)
all_group_papers["is_cross"] = all_group_papers["n_crossgroup_authors"] > 0
group_totals = all_group_papers.groupby("group")["paper"].nunique()

PANEL_LABELS = {g: chr(ord("a") + i) for i, g in enumerate(GROUPS)}

fig, axes = plt.subplots(1, 4, figsize=(16, 5.2), sharey=True)
for ax, group in zip(axes, GROUPS):
    sub_all = all_group_papers[all_group_papers["group"] == group]
    sub_non = sub_all[~sub_all["is_cross"]]
    sub_cross = sub_all[sub_all["is_cross"]]
    n_total = int(group_totals.get(group, 0))
    n_cross = len(sub_cross)
    pct = (n_cross / n_total * 100) if n_total else 0.0

    for yi in range(len(LEVEL_LABELS)):
        ax.add_patch(Rectangle(
            (-0.6, yi - BAND_HALF), len(CRITERIA) - 0.4 + 0.6, 2 * BAND_HALF,
            facecolor=BAND_COLOR, edgecolor="none", alpha=0.35, zorder=0,
        ))
    for xi in range(len(CRITERIA)):
        ax.add_patch(Rectangle(
            (xi - BAND_HALF, -0.6), 2 * BAND_HALF, len(LEVEL_LABELS) - 0.4 + 0.6,
            facecolor=BAND_COLOR, edgecolor="none", alpha=0.35, zorder=0,
        ))

    def _jitter_subset(sub, rng):
        coords = {}
        for _, row in sub.iterrows():
            pts = []
            for xi, crit in enumerate(CRITERIA):
                val = row[f"consolidated_{crit}"]
                if pd.isna(val) or val not in LEVEL_LABELS:
                    pts.append(None)
                else:
                    yi = LEVEL_LABELS.index(val)
                    x = xi + rng.uniform(-X_JIT, X_JIT)
                    y = yi + rng.uniform(-Y_JIT, Y_JIT)
                    pts.append((x, y))
            coords[row["paper"]] = pts
        return coords

    coords_non = _jitter_subset(sub_non, rng)
    coords_cross = _jitter_subset(sub_cross, rng)

    for coords, color, alpha_line in [
        (coords_non,   COLOR_NONCROSS,        0.18),
        (coords_cross, GROUP_COLORS[group],    0.35),
    ]:
        for paper, pts in coords.items():
            valid = [(x, y) for p in pts if p is not None for x, y in [p]]
            if len(valid) >= 2:
                xs, ys = zip(*valid)
                ax.plot(xs, ys, color=color, linewidth=0.6, alpha=alpha_line, zorder=1)

    for paper, pts in coords_non.items():
        for pt in pts:
            if pt is None:
                continue
            ax.scatter(pt[0], pt[1], s=SIZE_BASE, alpha=0.45,
                       facecolor=COLOR_NONCROSS, edgecolor="white", linewidth=0.3,
                       zorder=2)

    for paper, pts in coords_cross.items():
        n_auth = int(sub_cross.loc[sub_cross["paper"] == paper, "n_crossgroup_authors"].iloc[0])
        s = SIZE_BASE + SIZE_PER_AUTHOR * (n_auth - 1)
        for pt in pts:
            if pt is None:
                continue
            ax.scatter(pt[0], pt[1], s=s, alpha=0.7,
                       facecolor=GROUP_COLORS[group], edgecolor="white", linewidth=0.4,
                       zorder=4)

    label = PANEL_LABELS[group]
    ax.set_title(
        f"({label}) {GROUP_TITLES[group]}",
        fontsize=10, fontweight="bold", pad=14,
    )
    ax.text(
        0.5, 1.005,
        f"{n_cross}/{n_total} papers have a cross-group author ({pct:.0f}%)",
        transform=ax.transAxes, fontsize=9, ha="center", va="bottom",
    )

    ax.set_xticks(range(len(CRITERIA)))
    ax.set_xticklabels([CRIT_LABELS[c] for c in CRITERIA])
    ax.set_xlim(-0.6, len(CRITERIA) - 0.4)
    ax.set_ylim(-0.6, len(LEVEL_LABELS) - 0.4)
    ax.tick_params(axis="both", length=0)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

axes[0].set_yticks(range(len(LEVEL_LABELS)))
axes[0].set_yticklabels(LEVEL_LABELS)
axes[0].set_ylabel("Reproducibility level (achieved)")

legend_handles = [
    Line2D([0], [0], marker="o", linestyle="", markersize=SIZE_BASE**0.5,
           markerfacecolor=COLOR_NONCROSS, markeredgecolor="white",
           alpha=0.6, label="No cross-group author"),
    Line2D([0], [0], marker="o", linestyle="", markersize=SIZE_BASE**0.5,
           markerfacecolor="0.25", markeredgecolor="white",
           label="1 cross-group author (panel colour)"),
]
for n_auth in (2, 3, 4):
    legend_handles.append(
        Line2D([0], [0], marker="o", linestyle="",
               markersize=(SIZE_BASE + SIZE_PER_AUTHOR * (n_auth - 1))**0.5,
               markerfacecolor="0.45", markeredgecolor="white",
               label=f"{n_auth} cross-group authors"),
    )
axes[2].legend(handles=legend_handles, loc="upper right", fontsize=7.5,
               frameon=False, labelspacing=1.2, borderpad=1.0)

fig.suptitle(
    "Reproducibility across the four analytical groups\n"
    "One point per paper (jittered). Grey = no cross-group author; "
    "coloured = >=1 cross-group author (size scales with the count).\n"
    "Thin lines connect each paper's three criterion values. "
    "2020 excluded (AGILE transition year).",
    fontsize=11,
)
fig.tight_layout()
fig.subplots_adjust(top=0.82)
fig.savefig(FIG_DIR / "figure_crossgroup_overlap_reprolevels.png", dpi=150)
fig.savefig(FIG_DIR / "figure_crossgroup_overlap_reprolevels.pdf")
plt.show()

## 5. Cross-group author overlap (non-independence check)

The main hypothesis testing in [`04_results_hypotheses.ipynb`](04_results_hypotheses.ipynb)
(Mann-Whitney-U, rank-biserial effect sizes, Fisher exact tests; manuscript Tables 4-7)
treats the four groups as independent samples:

- `agile_pre`      -- AGILE papers,     `year < 2020`
- `agile_post`     -- AGILE papers,     `year > 2020`
- `giscience_pre`  -- GIScience papers, `year < 2020`
- `giscience_post` -- GIScience papers, `year > 2020`

Papers from 2020 are excluded, matching the rest of the analysis (AGILE transition year).

Independence assumes that papers in one group do not share authors with papers in another group (or within the same group). When the same person contributes to multiple papers across these groups, the groups are correlated and the apparent effect size of any pre-vs-post or AGILE-vs-GIScience comparison can be inflated.

**This section is a first iteration**: for every author with `>= 2` papers (irrespective of whether those papers cross a conference or time-period boundary) we list each of their papers and which of the four groups it belongs to, together with the reproducibility level recorded for each criterion. No aggregation -- just the raw values of the categorical variables that feed the main analysis.

In [ ]:
# Keep only authors with >= 2 papers (pooling across the 4 groups).
papers_per_author = grouped.groupby("identity")["paper"].nunique()
multi_ids = papers_per_author[papers_per_author > 1].index
multi = grouped[grouped["identity"].isin(multi_ids)].copy()

# Attach reproducibility levels from df so the raw categorical values are visible.
multi = multi.merge(
    df[["paper", "consolidated_data", "consolidated_methods", "consolidated_results"]],
    on="paper", how="left",
)

multi = multi.sort_values(["identity", "year", "conf", "paper"]).reset_index(drop=True)

n_authors = multi["identity"].nunique()
n_rows = len(multi)
groups_per_author = (
    multi.groupby("identity")["group"].nunique().value_counts().sort_index()
)
print(f"multi-paper authors: {n_authors}")
print(f"author-paper rows for those authors: {n_rows}")
print()
print("count of authors by number of distinct groups they appear in:")
for k, v in groups_per_author.items():
    print(f"  appears in {k} group(s): {v} author(s)")
print()
print("papers per group (restricted to multi-paper authors):")
print(multi.groupby("group")["paper"].nunique().to_string())

cols = ["identity", "name", "group", "conf", "year", "paper",
        "consolidated_data", "consolidated_methods", "consolidated_results"]
out_path = "data/cross-group-overlap-multipaper-authors.csv"
multi[cols].to_csv(out_path, index=False)
print()
print(f"wrote {out_path} ({n_rows} rows)")

with pd.option_context("display.max_rows", None, "display.max_colwidth", 60):
    display(multi[cols])

## 6. Persist outputs

All outputs produced by this notebook land in [`data/`](data/) (regenerable artefacts):

- `authors-cross-conference-yearly.csv` — per-year same-year overlap counts (B.1).
- `cross-author-reprolevels-yearly.csv` — per-(year, level, subset) reproducibility counts (4.1).
- `cross-author-intervention-delta.csv` — pre-vs-post %HIGH delta for all vs. cross-author papers (4.2).
- `cross-group-overlap-multipaper-authors.csv` — raw author-paper table for the non-independence check (5).

In [ ]:
# B.1 per-year overlap counts (yearly_overlap was built in the Part B cell above).
yearly_overlap.to_csv("data/authors-cross-conference-yearly.csv", index=False)
print("wrote data/authors-cross-conference-yearly.csv")
# All other outputs (cross-author-reprolevels-yearly.csv, cross-author-intervention-delta.csv,
# cross-group-overlap-multipaper-authors.csv) are written inline by their respective cells above.